# FlareWise — Fine-tune DistilBERT on symptom→domain (combined + rigorous)

Fine-tunes a **pretrained DistilBERT** transformer on a **combined** symptom corpus
(`gretelai/symptom_to_diagnosis` + `NeuronZero/Symptom2Disease`), mapped into the
7 FlareWise clinical domains. Trains with **class weighting + early stopping**,
compares head-to-head against a classical TF-IDF + Logistic Regression baseline
using **accuracy, macro-F1, and per-class F1**, runs a harder **intake-style
generalization test**, and exports to ONNX for the web app (Transformers.js).

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
~10-15 min on a free T4. A zip downloads at the end.

> Note on data size: public symptom→diagnosis text datasets mostly trace back to the
> same ~1,200 Kaggle examples. Combining the two natural-language variants roughly
> doubles the data (~2,000+ rows) and adds phrasing diversity; the bigger wins here
> are the class weighting, early stopping, and honest per-class evaluation.

In [ ]:
# 1. Install
!pip -q install "transformers>=4.44" "datasets>=2.20" "evaluate" "scikit-learn" "accelerate>=0.30" "optimum[onnxruntime]"
print("done")

In [ ]:
# 2. Load + combine + map to 7 FlareWise domains
from datasets import load_dataset

DIAGNOSIS_TO_DOMAIN = {
    "allergy": "respiratory_or_immune", "arthritis": "musculoskeletal",
    "bronchial asthma": "respiratory_or_immune", "cervical spondylosis": "musculoskeletal",
    "chicken pox": "infection_or_fever", "common cold": "respiratory_or_immune",
    "dengue": "infection_or_fever", "diabetes": "metabolic_or_systemic",
    "dimorphic hemorrhoids": "gastrointestinal_or_urinary", "drug reaction": "skin_or_medication_reaction",
    "fungal infection": "skin_or_medication_reaction", "gastroesophageal reflux disease": "gastrointestinal_or_urinary",
    "hypertension": "cardiovascular_or_neurologic", "impetigo": "skin_or_medication_reaction",
    "jaundice": "metabolic_or_systemic", "malaria": "infection_or_fever",
    "migraine": "cardiovascular_or_neurologic", "peptic ulcer disease": "gastrointestinal_or_urinary",
    "pneumonia": "respiratory_or_immune", "psoriasis": "skin_or_medication_reaction",
    "typhoid": "infection_or_fever", "urinary tract infection": "gastrointestinal_or_urinary",
    "varicose veins": "cardiovascular_or_neurologic",
    "acne": "skin_or_medication_reaction",   # extra class present in NeuronZero
}

import re
def norm(t): return re.sub(r"\s+", " ", str(t).strip().lower())
def to_dom(diag): return DIAGNOSIS_TO_DOMAIN.get(norm(diag))

# Canonical held-out TEST = gretel's official test split (keeps results comparable
# to the in-app model's reported numbers). Everything else becomes training pool.
g = load_dataset("gretelai/symptom_to_diagnosis")
gcols = g["train"].column_names
gt = "input_text" if "input_text" in gcols else gcols[0]
gl = "output_text" if "output_text" in gcols else gcols[1]

def rows_from(dataset, split, tcol, lcol):
    out, skip = [], set()
    for r in dataset[split]:
        d = to_dom(r[lcol])
        if d is None: skip.add(r[lcol]); continue
        out.append((norm(r[tcol]), str(r[tcol]).strip(), d))
    if skip: print(f"  skipped unmapped in {split}:", skip)
    return out

test_rows = rows_from(g, "test", gt, gl)
train_pool = rows_from(g, "train", gt, gl)

# Add NeuronZero (Kaggle original, natural language) to the TRAINING pool only.
try:
    n = load_dataset("NeuronZero/Symptom2Disease")
    nsplit = "train" if "train" in n else list(n.keys())[0]
    ncols = n[nsplit].column_names
    nt = "text" if "text" in ncols else [c for c in ncols if c != "label"][-1]
    nl = "label" if "label" in ncols else ncols[0]
    train_pool += rows_from(n, nsplit, nt, nl)
    print("added NeuronZero rows")
except Exception as e:
    print("NeuronZero unavailable, continuing with gretel only:", e)

# Dedup training against the test set (shared Kaggle origin -> leakage risk).
test_norms = {nrm for nrm, _, _ in test_rows}
seen, train_rows = set(), []
for nrm, raw, dom in train_pool:
    if nrm in test_norms or nrm in seen: continue
    seen.add(nrm); train_rows.append((nrm, raw, dom))

train_texts = [r for _, r, _ in train_rows]; train_labels = [d for _, _, d in train_rows]
test_texts  = [r for _, r, _ in test_rows];  test_labels  = [d for _, _, d in test_rows]
labels = sorted(set(train_labels) | set(test_labels))
label2id = {l: i for i, l in enumerate(labels)}; id2label = {i: l for l, i in label2id.items()}

from collections import Counter
print(f"train={len(train_texts)}  test={len(test_texts)}  classes={len(labels)}")
print("train class balance:", dict(Counter(train_labels)))

In [ ]:
# 3. Classical baseline: TF-IDF + Logistic Regression (class-weighted)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
Xtr, Xte = vec.fit_transform(train_texts), vec.transform(test_texts)
lr = LogisticRegression(max_iter=3000, C=4.0, class_weight="balanced")
lr.fit(Xtr, train_labels)
pred = lr.predict(Xte)
base_acc = accuracy_score(test_labels, pred)
base_f1 = f1_score(test_labels, pred, average="macro")
print(f"TF-IDF + LR   accuracy={base_acc:.3f}   macro-F1={base_f1:.3f}\n")
print(classification_report(test_labels, pred, digits=3))

In [ ]:
# 4. Fine-tune DistilBERT with class weighting + early stopping
import numpy as np, torch
from torch import nn
from datasets import Dataset
from sklearn.utils.class_weight import compute_class_weight
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding,
                          EarlyStoppingCallback)

MODEL = "distilbert-base-uncased"
tok = AutoTokenizer.from_pretrained(MODEL)
def mk(texts, labs):
    d = Dataset.from_dict({"text": texts, "label": [label2id[l] for l in labs]})
    return d.map(lambda b: tok(b["text"], truncation=True, max_length=128), batched=True)
train_tok, test_tok = mk(train_texts, train_labels), mk(test_texts, test_labels)

cw = compute_class_weight("balanced", classes=np.arange(len(labels)),
                          y=[label2id[l] for l in train_labels])
class_weights = torch.tensor(cw, dtype=torch.float)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=len(labels), id2label=id2label, label2id=label2id)

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {"accuracy": accuracy_score(p.label_ids, preds),
            "macro_f1": f1_score(p.label_ids, preds, average="macro")}

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels_ = inputs.pop("labels")
        out = model(**inputs)
        loss = nn.functional.cross_entropy(
            out.logits, labels_, weight=class_weights.to(out.logits.device))
        return (loss, out) if return_outputs else loss

common = dict(output_dir="out", num_train_epochs=8,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    learning_rate=3e-5, save_strategy="epoch", logging_steps=20,
    load_best_model_at_end=True, metric_for_best_model="macro_f1",
    report_to="none")
try:
    args = TrainingArguments(eval_strategy="epoch", **common)
except TypeError:
    args = TrainingArguments(evaluation_strategy="epoch", **common)

trainer = WeightedTrainer(model=model, args=args, train_dataset=train_tok,
    eval_dataset=test_tok, tokenizer=tok, data_collator=DataCollatorWithPadding(tok),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
trainer.train()
m = trainer.evaluate(); print(m)

In [ ]:
# 5. Head-to-head + per-class report for DistilBERT
import numpy as np
from sklearn.metrics import classification_report
bert_acc, bert_f1 = m["eval_accuracy"], m["eval_macro_f1"]
preds = np.argmax(trainer.predict(test_tok).predictions, axis=1)
bert_pred = [id2label[i] for i in preds]

print(f"{'Method':<30}{'Accuracy':>11}{'Macro-F1':>11}")
print("-"*52)
print(f"{'TF-IDF + LogReg (classical)':<30}{base_acc:>11.3f}{base_f1:>11.3f}")
print(f"{'DistilBERT (fine-tuned)':<30}{bert_acc:>11.3f}{bert_f1:>11.3f}")
print(f"{'Delta (BERT - classical)':<30}{bert_acc-base_acc:>11.3f}{bert_f1-base_f1:>11.3f}")
print("\nDistilBERT per-class:\n")
print(classification_report(test_labels, bert_pred, digits=3))

In [ ]:
# 6. Harder generalization: messy multi-symptom intake notes (unseen style)
INTAKE_EVAL = [
    ("Headaches and dizziness for three days, pain about 6 out of 10. Tried ibuprofen with no relief.", "cardiovascular_or_neurologic"),
    ("My blood pressure has been high. I get dizzy in the mornings. Started new BP meds last month.", "cardiovascular_or_neurologic"),
    ("Migraine behind the eyes with blurred vision. Tried Tylenol.", "cardiovascular_or_neurologic"),
    ("Severe abdominal pain and nausea, vomiting twice today. Took Tums but it didn't help.", "gastrointestinal_or_urinary"),
    ("Burning when urinating for two days. Drinking more water.", "gastrointestinal_or_urinary"),
    ("Heartburn after meals, sour taste in mouth. Tried over the counter antacids.", "gastrointestinal_or_urinary"),
    ("Dry cough and runny nose for a week. Used a humidifier and rested.", "respiratory_or_immune"),
    ("Wheezing and shortness of breath. Using my inhaler more than usual.", "respiratory_or_immune"),
    ("Sneezing and watery eyes whenever I go outside. Tried Benadryl.", "respiratory_or_immune"),
    ("Rash on my arms and itching after starting a new medication.", "skin_or_medication_reaction"),
    ("Red scaly patches on my elbow. Used a moisturizer.", "skin_or_medication_reaction"),
    ("Joint pain and stiffness in my knees, worse in the morning. Tried Advil.", "musculoskeletal"),
    ("Lower back pain after lifting something heavy. Used a heating pad.", "musculoskeletal"),
    ("Neck pain and stiffness for a week, hard to turn my head.", "musculoskeletal"),
    ("High fever and chills for two days, body aches. Took acetaminophen.", "infection_or_fever"),
    ("Sore throat with fever and swollen glands. Drinking lots of fluids.", "infection_or_fever"),
    ("Excessive thirst and frequent urination. Tired all the time.", "metabolic_or_systemic"),
    ("Yellowing of the eyes and skin, dark urine.", "metabolic_or_systemic"),
]
from transformers import pipeline as hf_pipeline
pipe = hf_pipeline("text-classification", model=model, tokenizer=tok,
                   device=0 if torch.cuda.is_available() else -1)
te = [t for t,_ in INTAKE_EVAL]; gold = [g for _,g in INTAKE_EVAL]
b = [pipe(t)[0]["label"] for t in te]; l = list(lr.predict(vec.transform(te)))
print(f"Intake-style generalization:  TF-IDF+LR={sum(p==g for p,g in zip(l,gold))/len(gold):.3f}"
      f"   DistilBERT={sum(p==g for p,g in zip(b,gold))/len(gold):.3f}")

In [ ]:
# 7. Export to ONNX for the FlareWise app (Transformers.js)
import os, shutil
SAVE, OUT = "ft-model", "distilbert-symptom-domain"
model.save_pretrained(SAVE); tok.save_pretrained(SAVE)
from optimum.onnxruntime import ORTModelForSequenceClassification
ort = ORTModelForSequenceClassification.from_pretrained(SAVE, export=True)
os.makedirs(OUT + "/onnx", exist_ok=True)
ort.save_pretrained(OUT); tok.save_pretrained(OUT)
if os.path.exists(OUT + "/model.onnx"):
    shutil.move(OUT + "/model.onnx", OUT + "/onnx/model.onnx")
try:
    from optimum.onnxruntime import ORTQuantizer
    from optimum.onnxruntime.configuration import AutoQuantizationConfig
    qz = ORTQuantizer.from_pretrained(OUT, file_name="onnx/model.onnx")
    qz.quantize(save_dir=OUT + "/onnx",
                quantization_config=AutoQuantizationConfig.avx512_vnni(is_static=False, per_channel=False))
    print("quantized model written")
except Exception as e:
    print("quantization skipped:", e)
shutil.make_archive(OUT, "zip", OUT)
print("contents:", os.listdir(OUT), "| onnx:", os.listdir(OUT + "/onnx"))
from google.colab import files
files.download(OUT + ".zip")